# Milestone 2 Exploration  (Internal Only)

This notebook is to confirm our .py script outputs, not for submission.

In [1]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [2]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [3]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   parent_asin                    20000 non-null  str    
 1   product_title                  20000 non-null  str    
 2   features                       20000 non-null  object 
 3   description                    20000 non-null  object 
 4   categories                     20000 non-null  object 
 5   details                        20000 non-null  object 
 6   price                          11212 non-null  float64
 7   derived_avg_rating             8669 non-null   float64
 8   n_reviews                      20000 non-null  int64  
 9   review_text                    8669 non-null   str    
 10  candidate_review_title         8669 non-null   str    
 11  candidate_review_text          8669 non-null   str    
 12  candidate_review_helpful_vote  8669 non-null   float64
dt

In [5]:
import altair as alt

df["review_text_len"] = df["review_text"].str.len()

df.describe()

,price,derived_avg_rating,n_reviews,candidate_review_helpful_vote,review_text_len
count,11212.000000,8669.000000,20000.00000,8669.000000,8669.000000
mean,88.479685,4.256767,3.53030,5.083977,1777.333949
std,310.481798,1.092489,23.69879,38.984341,7083.271815
min,0.010000,1.000000,0.00000,0.000000,6.000000
25%,14.990000,4.000000,0.00000,0.000000,131.000000
50%,26.990000,4.750000,0.00000,0.000000,370.000000
75%,59.950000,5.000000,1.00000,2.000000,1074.000000
max,7691.010000,5.000000,1220.00000,1835.000000,209547.000000


In [6]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,n_reviews,review_text,candidate_review_title,candidate_review_text,candidate_review_helpful_vote,review_text_len
0,B008YDSH6E,"Supco RCO410 Start Kit , BLACK","[For compressor sizes 1/4 through 1/3 HP, Comb...","[Product Description, Are you looking to repla...","[Appliances, Parts & Accessories, Refrigerator...","[(Brand, ""Supco""), (Voltage, ""288 Volts""), (Po...",13.95,4.657895,38,Supco RCO410 Start Kit: This Supco RCO410 Star...,$3000 side by side saved,10 year old Kenmore side by side freezer only ...,4.0,17329.0
1,B0C7GKM91B,"EcoAqua EFF-6027A Replacement Filter, Compatib...",[【CERTIFIED AND TESTED】: Tested and certified ...,[],"[Appliances, Parts & Accessories, Refrigerator...","[(Material, ""Lead-Free Material, Food-Grade Ma...",11.09,4.443114,334,"great deal: This was a great deal, good price ...",NOT THE SAME....SEE PHOTO Did not work in my ...,DID NOT WORK! Bought to replace DA29-00020B f...,134.0,60267.0
2,B005VH6WD6,Vent Kit Dryer Plst 4inx7ft Br,"[Louver Vent Kit, Size: 4"" x 7', Hood Color: B...","[Includes: 1 - louvered vent with tail pipe, 1...","[Appliances, Parts & Accessories, Dryer Parts ...","[(Product Dimensions, ""30 x 13 x 13 inches""), ...",26.19,3.000000,1,Shorted two feet of flex: Cheap plastic flex t...,Shorted two feet of flex,Cheap plastic flex that resembled metal is 5 f...,0.0,106.0
3,B07QXRXXPJ,WeTest Silicone Kitchen Stove Counter Gap Cove...,[Extra-clear - no opaque - multi-purpose stove...,[],"[Appliances, Parts & Accessories, Range Parts ...","[(Manufacturer, ""WeTest""), (Brand, ""WeTest""), ...",NaN,5.000000,1,Dang this time.: Have used and loved in the pa...,Dang this time.,Have used and loved in the past. Unfortunately...,0.0,147.0
4,B089VSYR3T,My K Cup Universal Reusable Coffee Pods Filter...,[√ COMPATIBLE WITH MODELS SPECIFIED FIT FOR KE...,[],"[Small Appliance Parts & Accessories, Coffee &...","[(Package Dimensions, ""5.51 x 3.03 x 3.03 inch...",11.88,3.772727,22,Perfect! Work great: Works perfect with our Ke...,Doesn’t work for me,Since this cup resembles the actual cup from K...,10.0,2987.0


In [7]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
review_text_len,11331,0.56655
candidate_review_helpful_vote,11331,0.56655
candidate_review_title,11331,0.56655
candidate_review_text,11331,0.56655
review_text,11331,0.56655
derived_avg_rating,11331,0.56655
price,8788,0.43940
categories,0,0.00000
details,0,0.00000
features,0,0.00000


In [8]:
df.describe()

,price,derived_avg_rating,n_reviews,candidate_review_helpful_vote,review_text_len
count,11212.000000,8669.000000,20000.00000,8669.000000,8669.000000
mean,88.479685,4.256767,3.53030,5.083977,1777.333949
std,310.481798,1.092489,23.69879,38.984341,7083.271815
min,0.010000,1.000000,0.00000,0.000000,6.000000
25%,14.990000,4.000000,0.00000,0.000000,131.000000
50%,26.990000,4.750000,0.00000,0.000000,370.000000
75%,59.950000,5.000000,1.00000,2.000000,1074.000000
max,7691.010000,5.000000,1220.00000,1835.000000,209547.000000


## Debugging Code

The code below were explorations to help inform the pipeline and find bugs. Codex was used to generate quick debugging code below.

In [9]:
def present(s):
    return s.notna() & s.astype(str).str.strip().ne("")

df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

summary = pd.crosstab(
    index=[
        df["has_n_reviews"],
        df["has_avg_rating"],
        df["has_candidate_title"],
        df["has_candidate_text"],
    ],
    columns="count",
).reset_index()

summary = summary.sort_values("count", ascending=False)
summary

col_0,has_n_reviews,has_avg_rating,has_candidate_title,has_candidate_text,count
0,False,False,False,False,11331
2,True,True,True,True,8668
1,True,True,True,False,1


In [10]:
# Review count exists, but no candidate review shown
df.loc[
    (df["n_reviews"] > 0)
    & (~present(df["candidate_review_title"]))
    & (~present(df["candidate_review_text"])),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
].head(20)

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote


In [11]:
def present(s):
    cleaned = s.astype("string").str.strip()
    missing_tokens = {"", "na", "n/a", "nan", "none", "null"}
    return cleaned.notna() & ~cleaned.str.lower().isin(missing_tokens)

In [12]:
df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

In [13]:
df.loc[
    df["product_title"].str.contains(
        "Pour Over Coffee Dripper Stainless Steel Double Layer Mesh",
        regex=False,
        na=False,
    ),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
19778,B07D6L2QX2,Pour Over Coffee Dripper Stainless Steel Doubl...,0,NaN,NaN,NaN,NaN


In [14]:
df.loc[
    df["parent_asin"] == "B09VT3BS1G",
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
13203,B09VT3BS1G,"MoMoSun Furniture Dolly,Extendable Washing Mac...",0,NaN,NaN,NaN,NaN
